# Colab v2 — TinyLlama batching benchmark

**Runtime → A100 GPU.** Upload `InferenceLab_v2.zip`, run all cells.


In [ ]:
!nvidia-smi


In [ ]:
import os
os.chdir('/content')
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))


In [ ]:
import os, shutil, zipfile, glob

os.chdir('/content')
PROJECT_ROOT = '/content/inference_lab'
shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
os.makedirs(PROJECT_ROOT)

zip_name = 'InferenceLab_v2.zip'
if not os.path.exists(zip_name):
    zips = sorted(glob.glob('/content/*.zip'), key=os.path.getmtime)
    assert zips, 'Upload InferenceLab_v2.zip first'
    zip_name = zips[-1]

with zipfile.ZipFile(zip_name) as z:
    z.extractall(PROJECT_ROOT)

assert os.path.isdir(f'{PROJECT_ROOT}/server')
os.chdir(PROJECT_ROOT)
print('Working dir:', os.getcwd())


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import json, subprocess, time, sys, os, urllib.request

PROJECT_ROOT = '/content/inference_lab'
os.chdir(PROJECT_ROOT)

def wait_for_server(server, timeout=600):
    start = time.time()
    while time.time() - start < timeout:
        if server.poll() is not None:
            return False
        try:
            with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5) as r:
                if json.loads(r.read()).get('status') == 'ok':
                    return True
        except Exception:
            pass
        time.sleep(2)
    return False

def run_strategy(app_module, strategy, runs=3):
    env = os.environ.copy()
    env['PYTHONPATH'] = PROJECT_ROOT
    server = subprocess.Popen(
        ['uvicorn', app_module, '--host', '127.0.0.1', '--port', '8000'],
        env=env, cwd=PROJECT_ROOT,
    )
    try:
        assert wait_for_server(server), f'{strategy} server failed'
        cmd = [sys.executable, 'scripts/run_benchmark_suite.py',
               '--url', 'http://127.0.0.1:8000', '--strategy', strategy,
               '--runs', str(runs), '--monitor-gpu']
        assert subprocess.run(cmd, env=env, cwd=PROJECT_ROOT).returncode == 0
    finally:
        server.terminate()
        server.wait(timeout=15)
        time.sleep(3)

print('Ready:', PROJECT_ROOT)


In [ ]:
run_strategy('server.main:app', 'baseline')
run_strategy('server.batched_server:app', 'batched')
run_strategy('server.dynamic_server:app', 'dynamic')


In [ ]:
import os
os.chdir('/content/inference_lab')
os.makedirs('report', exist_ok=True)
!python scripts/plot_tail_latency.py --results-dir results --out-dir report --tier v2
!python scripts/generate_tier_charts.py


In [ ]:
import os
os.chdir('/content/inference_lab')
!zip -r results_and_report.zip results report
from google.colab import files
files.download('results_and_report.zip')
